In [6]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import pearsonr, f_oneway
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.cluster import KMeans
from sentence_transformers import SentenceTransformer
from transformers import pipeline
import xgboost as xgb
from sklearn.metrics import mean_squared_error, r2_score
import logging
from diffprivlib.models import GaussianNB  # Simulated differential privacy
from joblib import dump
from reportlab.lib.pagesizes import letter
from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer, Image, Table, TableStyle
from reportlab.lib.styles import getSampleStyleSheet
from reportlab.lib import colors

In [7]:
# Set up logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

In [9]:
# Privacy-awareData Loading
def load_data_privacy_aware(file_path):
    logging.info("Loading data with privacy simulation")
    df = pd.read_csv(file_path)
    # Simulate differential privacy noise on numeric columns
    numeric_cols = ['Views', 'Likes', 'Shares', 'Comments']
    for col in numeric_cols:
        noise = np.random.laplace(0, 0.1, size=len(df))  # Small noise for demo
        df[col] = df[col] + noise
        df[col] = df[col].clip(lower=0)  # Ensure non-negative
    return df

In [10]:
# Data Pipeline
def process_data(df):
    logging.info("Processing data in pipeline")
    df.dropna(inplace=True)
    for col in ['Platform', 'Hashtag', 'Content_Type', 'Region', 'Engagement_Level']:
        df[col] = df[col].astype('category')
    numeric_cols = ['Views', 'Likes', 'Shares', 'Comments']
    scaler = StandardScaler()
    df[numeric_cols] = scaler.fit_transform(df[numeric_cols])
    df['Engagement_Score'] = df[numeric_cols].mean(axis=1)
    df['Log_Views'] = np.log1p(df['Views'].clip(lower=0))
    return df, scaler

In [11]:
# LLM Features
def add_llm_features(df):
    logging.info("Adding LLM features")
    model = SentenceTransformer('all-MiniLM-L6-v2')
    embeddings = model.encode(df['Hashtag'].tolist(), show_progress_bar=True)
    kmeans = KMeans(n_clusters=3, random_state=42)
    df['Hashtag_Cluster'] = kmeans.fit_predict(embeddings)
    cluster_names = {0: 'Trendy', 1: 'Informative', 2: 'Casual'}
    df['Cluster_Name'] = df['Hashtag_Cluster'].map(cluster_names)
    
    sentiment_analyzer = pipeline("sentiment-analysis", model="distilbert-base-uncased-finetuned-sst-2-english")
    df['Hashtag_Sentiment'] = df['Hashtag'].apply(
        lambda x: sentiment_analyzer(x)[0]['score'] if sentiment_analyzer(x)[0]['label'] == 'POSITIVE' else -sentiment_analyzer(x)[0]['score']
    )
    top_hashtags = df.groupby('Platform').apply(lambda x: x.loc[x['Engagement_Score'].idxmax(), 'Hashtag']).to_dict()
    df['Top_Hashtag_Platform'] = df['Platform'].map(top_hashtags)
    return df

In [13]:
# Predictive Modeling with XGBoost
def train_predictive_model(df):
    logging.info("Training XGBoost model for Engagement_Score prediction")
    features = ['Views', 'Likes', 'Shares', 'Comments', 'Hashtag_Sentiment', 'Hashtag_Cluster']
    X = df[features]
    y = df['Engagement_Score']
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
    
    xgb_model = xgb.XGBRegressor(n_estimators=100, random_state=42, objective='reg:squarederror')
    xgb_model.fit(X_train, y_train)
    y_pred = xgb_model.predict(X_test)
    
    mse = mean_squared_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)
    feature_importance = pd.Series(xgb_model.feature_importances_, index=features).sort_values(ascending=False)
    logging.info(f"Model Performance - MSE: {mse:.4f}, R2: {r2:.4f}")
    logging.info(f"Feature Importance:\n{feature_importance}")

    # Simulate Core ML export
    dump(xgb_model, 'xgb_model.joblib')
    logging.info("Model saved as 'xgb_model.joblib' (simulating Core ML export)")
    return xgb_model, mse, r2, feature_importance

In [18]:
# Visualization Generation (Minimalist Design)
def generate_visualizations(df):
    logging.info("Generating visualizations with Apple-inspired design")
    sns.set_style("white")  # Clean, minimalist look
    plt.rcParams['font.family'] = 'Helvetica'
    # Bar Chart
    platform_eng = df.groupby('Platform').agg({'Engagement_Score': 'mean'}).reset_index()
    plt.figure(figsize=(10, 6))
    sns.barplot(data=platform_eng, x='Platform', y='Engagement_Score', palette='colorblind')
    plt.title("Engagement by Platform", fontsize=16, pad=20)
    plt.xlabel(""); plt.ylabel("")  # Minimal labels
    plt.savefig("engagement_by_platform.png", dpi=300, bbox_inches='tight')
    plt.show() 
    # Pie Chart
    content_dist = df['Content_Type'].value_counts()
    plt.figure(figsize=(8, 8))
    plt.pie(content_dist, labels=content_dist.index, autopct='%1.1f%%', colors=sns.color_palette('colorblind', n_colors=len(content_dist)), 
            textprops={'fontsize': 12})
    plt.title("Content Type Distribution", fontsize=16, pad=20)
    plt.savefig("content_type_distribution.png", dpi=300, bbox_inches='tight')
    plt.show()
    # Grouped Bar Chart
    plt.figure(figsize=(10, 6))
    sns.barplot(data=df, x='Engagement_Level', y='Hashtag_Sentiment', hue='Platform', palette='colorblind')
    plt.title("Sentiment by Engagement", fontsize=16, pad=20)
    plt.xlabel(""); plt.ylabel("")
    plt.legend().remove()  # Minimalist: remove legend if context clear
    plt.savefig("sentiment_by_engagement.png", dpi=300, bbox_inches='tight')
    plt.show()

    # Line Chart
    df['Index'] = range(len(df))
    plt.figure(figsize=(12, 6))
    plt.plot(df['Index'], df['Engagement_Score'], color=sns.color_palette('colorblind')[0], label='Actual')
    plt.plot(df['Index'], df['Trend_Prediction'], color=sns.color_palette('colorblind')[1], label='Trend')
    plt.title("Engagement Trend", fontsize=16, pad=20)
    plt.xlabel(""); plt.ylabel("")
    plt.legend(frameon=False)
    plt.savefig("engagement_trend.png", dpi=300, bbox_inches='tight')
    plt.show()

    # Heatmap
    plt.figure(figsize=(10, 6))
    pivot = df.pivot_table(values='Engagement_Score', index='Region', columns='Cluster_Name', aggfunc='mean')
    sns.heatmap(pivot, cmap='Blues', annot=True, fmt='.2f', cbar=False, annot_kws={"size": 12})
    plt.title("Engagement by Region & Cluster", fontsize=16, pad=20)
    plt.xlabel(""); plt.ylabel("")
    plt.savefig("engagement_heatmap.png", dpi=300, bbox_inches='tight')
    plt.show()

In [ ]:
# Bar Chart
    platform_eng = df.groupby('Platform').agg({'Engagement_Score': 'mean'}).reset_index()
    plt.figure(figsize=(10, 6))
    sns.barplot(data=platform_eng, x='Platform', y='Engagement_Score', palette='colorblind')
    plt.title("Engagement by Platform", fontsize=16, pad=20)
    plt.xlabel(""); plt.ylabel("")  # Minimal labels
    plt.savefig("engagement_by_platform.png", dpi=300, bbox_inches='tight')
    plt.close()